In [1]:
import pandas as pd
import numpy as np
import os

# get data with for loop


In [7]:

folder_path = r'D:\olist-ecommerce-analytics\data\raw'

df={}

for file_name in os.listdir(folder_path):
    if file_name.endswith('.csv'):
        file_path = os.path.join(folder_path, file_name)
        clean = file_name.replace('.csv', '')
        
        df[clean] = pd.read_csv(file_path)
    else:
        print(f"Skipping non-CSV file: {file_name}")    

Skipping non-CSV file: .gitkeep


In [3]:
df.keys()

dict_keys(['customers', 'olist_geolocation_dataset', 'olist_orders_dataset', 'olist_order_items_dataset', 'olist_order_payments_dataset', 'olist_order_reviews_dataset', 'olist_products_dataset', 'olist_sellers_dataset', 'product_category_name_translation'])

# Why is the `globals()` loop commented out?

### The Idea:
- When you have many tables (like `df['customers']`, `df['orders']`), it might seem helpful to extract them into standalone variables (`customers`, `orders`) so you don't have to type `df['...']` every time.

### Why is this a bad practice? (Why we don't uncomment it):
- **Hard to Debug:** It creates variables dynamically out of thin air. If you make a typo, it's very hard to track down where the variable came from or what it is.
- **Memory Heavy:** It duplicates references to large DataFrames in the global namespace, cluttering the memory.
- **Best Practice:** Keep them inside the `df` dictionary. Access them as `df['customers']`. It is cleaner and keeps all your data in one place.

In [ ]:
##for name , data in df.items():
    ##globals()[name] = data

# Create Function: `sum_nulls`

### Why `df.isnull().sum()`?
- `df.isnull()` returns a DataFrame of the exact same shape, but with `True` for empty cells and `False` for filled cells.
- Adding `.sum()` along the columns (axis 0 by default) adds up all the `True` values for each column. This instantly gives us the count of missing values per column without needing a loop.

### Why `len(df)` instead of `df.count()`?
- **Crucial point:** `df.count()` ignores `NaN` (missing) values and returns a smaller number if the dataset is incomplete.
- `len(df)` always returns the exact total number of rows in the DataFrame.
- To calculate a correct **percentage** of missing values, you must divide by the **total** number of rows (`len(df)`).

### Why `display()` instead of `print()`?
- In Jupyter Notebook, `display()` renders a clean, interactive HTML table.
- `print()` converts the DataFrame to a plain text string, which can cut off columns or look messy.

### Why are the values wrapped in square brackets `[count]`?
- When creating a DataFrame from a dictionary, Pandas usually expects values to be **lists** (columns).
- By wrapping the scalar values in `[ ]` (e.g., `[count]`), we force Pandas to treat them as a **single row** rather than trying to expand them into a long column.

### Why build it using a dictionary `{'count': ...}`?
- This standardizes the output format perfectly with your other helper functions (`sum_dup` and `sum_whitespace`), making the final `data_quality_report` clean and consistent.

In [ ]:
def sum_nulls(df):
     """
     Calculates the total count and percentage of null values for each column in a DataFrame.

     Parameters:
     df (pd.DataFrame): The input pandas DataFrame to analyze.

     Returns:
     pd.DataFrame: A DataFrame containing 'count' and 'null_percent' for each column.
     """
    
     count = df.isnull().sum()
     null_percent = (count / len(df)) * 100
     display(pd.DataFrame({'count': count, 'null_percent': null_percent}))

# Test Before Create

### Why `.duplicated().sum()` instead of `.duplicated().count()`?
- `duplicated()` returns a boolean **Series** (`True` for duplicates, `False` for unique).
- In Python, `True` is treated as `1` and `False` as `0`.
- Therefore, `.sum()` adds up all the `True` values, giving us exactly the number of duplicates.
- Using `.count()` would count *all* rows (including the non-duplicates), which is not what we want.

### Why does it return `np.int64(0)`?
- Pandas is built on NumPy, so it returns NumPy integer types. (It's perfectly fine, but you can wrap it in `int()` if you prefer a standard Python integer).

In [18]:
df['customers'].duplicated().sum()

np.int64(0)

# Create Master Function: `sum_dup`

### Why `.duplicated().sum()`?
- Counts the total number of duplicated rows efficiently.

### Why `len(df)` instead of `df.count()` for the percentage?
- **Crucial point:** `df.count()` ignores `NaN` (missing) values and returns a smaller number if the dataset is incomplete.
- `len(df)` always returns the exact total number of rows in the DataFrame. 
- To calculate a correct **percentage** of duplicates, you must divide by the **total** number of rows (`len(df)`).

### Why `display()` instead of `print()`?
- In Jupyter Notebook, `display()` renders a clean, interactive HTML table.
- `print()` converts the DataFrame to a plain text string, which can cut off columns or look messy.

### Why are the values wrapped in square brackets `[count]`?
- When creating a DataFrame from a dictionary, Pandas usually expects values to be **lists** (columns).
- By wrapping the scalar values in `[ ]` (e.g., `[count]`), we force Pandas to treat them as a **single row** rather than trying to expand them into a long column.

### Why build it using a dictionary `{'count': ...}`?
- This standardizes the output format perfectly with your other helper functions (`sum_whitespace` and `sum_nulls`), making the final `data_quality_report` clean and consistent.

In [ ]:
def sum_dup(df):
    """
    Calculates the total count and percentage of duplicated rows in a DataFrame.

    Parameters:
    df (pd.DataFrame): The input pandas DataFrame to analyze.

    Returns:
    pd.DataFrame: A DataFrame containing 'count' and 'dup_percent' for duplicated rows.
    """
    
    count = df.duplicated().sum()
    dup_percent = (count / len(df)) * 100
    display(pd.DataFrame({'count': [count], 'dup_percent': [dup_percent]}))

### test func

In [20]:
sum_dup(df['customers'])

,count,dup_percent
0,0,0.0


### here this is return whitespace something like trim in sql 

In [24]:
(df['customers']['customer_id'].str.strip() != df['customers']['customer_id']).sum()

np.int64(0)

# Create Function: `sum_whitespace`

### Why `df.select_dtypes(include=['object', 'string']).columns`?
- **Crucial point:** The `.str` accessor only works on text columns. If you try to run `.str.strip()` on a numeric column, Python will throw an `AttributeError`.
- `select_dtypes` filters the DataFrame *before* the loop, ensuring we only apply string operations to columns that actually contain strings (objects).

### Why `.fillna('')`?
- **Crucial point:** Pandas treats missing values (`NaN`) as `NaN`. If a cell is `NaN`, comparing `NaN != NaN` evaluates to `True`, which would incorrectly flag it as having "whitespace".
- By replacing `NaN` with an empty string `''`, the comparison `'' != ''` becomes `False`. This perfectly isolates only the cells that actually contain extra spaces.

### Why `.str.strip() != df[col_name]`?
- `strip()` removes leading and trailing spaces (e.g., `" Python "` becomes `"Python"`).
- We compare the stripped version against the original. If they are not equal `!=`, it means the original had extra spaces that were removed.
- **Alternative:** `df[col_name].str.strip()` is preferred over `df[col_name].str.replace(' ', '')` because `replace` removes *all* spaces (including spaces between words), while `strip` only removes the edges.

### Why `round(..., 2)`?
- It limits the percentage to 2 decimal places, making the final report much cleaner and easier to read.

### Why `int(white_space)` inside the dictionary?
- As we learned earlier, Pandas may coerce the `count` column to a Float if the `percent` column is a Float.
- Wrapping the count in `int()` ensures the count is displayed as a clean whole number (e.g., `5`) instead of `5.0`.

### Why `.T` at the end?
- When creating `pd.DataFrame(whitespace_counts)`, the column names become the keys, and the statistics (`count` and `percent`) become the index (rows).
- Using `.T` transposes the table. This is the standard, readable format for quality reports: **Column Name in the Index, and Count/Percent as Columns**.

In [ ]:
def sum_whitespace(df):
    
    
    whitespace_counts = {}
      
      
    """
    Calculates the total count and percentage of values with leading or trailing whitespace for each column in a DataFrame.

    Parameters:
    df (pd.DataFrame): The input pandas DataFrame to analyze.

    Returns:
    pd.DataFrame: A DataFrame containing 'count' and 'whitespace_percent' for each column.
    """
    
    
    for col_name in df.select_dtypes(include=['object', 'string']).columns:
        white_space = (df[col_name].fillna('').str.strip() != df[col_name].fillna('')).sum()
        white_space_percent = round((white_space / len(df)) * 100, 2)
        whitespace_counts[col_name] = {'count':int(white_space), 'whitespace_percent': white_space_percent}
    
    display(pd.DataFrame(whitespace_counts).T)
    





In [39]:
sum_whitespace(df['customers'])

                    count  whitespace_percent
customer_id           0.0                 0.0
customer_unique_id    0.0                 0.0
customer_city         0.0                 0.0
customer_state        0.0                 0.0


# Create Master Function: `data_quality_report`

### Why create a Master Function?
- **Modularity (DRY Principle):** Instead of manually calling `sum_dup`, `sum_whitespace`, and `sum_nulls` every single time for every table, we write a "wrapper" function that calls them all for us. This makes the notebook much cleaner and faster to run.
- **Consistency:** It ensures every dataframe gets the exact same quality checks applied in the exact same order every single time.

### Why `print(...)` for the headers and separators?
- Headers like `---shape---` and separators like `"="*50` are **strings**, not DataFrames. `display()` is specifically meant for rendering data tables (like our `sum_dup` function does). So we use `print` for text to keep it aligned.
- **The Separators (`"="*50`):** We use `print("\n" + "="*50 + "\n")` to create visual breaks between sections. The `"\n"` adds empty lines (spacing) to make the output less cluttered. The `"="*50` draws a line across the screen so you can easily see where one check ends and the next begins.

### Why `print(df.shape)`?
- `df.shape` returns a tuple (e.g., `(100, 5)`). Since it is the first line in the function, simply writing `df.shape` wouldn't show anything in the output. We MUST wrap it in `print()` if we want to see it!

### Why `df.info()` WITHOUT `print()`?
- **Crucial point:** The `df.info()` method in Pandas prints its output *directly to the console* and returns `None`. 
- If you wrote `print(df.info())`, Pandas would print the table, and then the `print` statement would print the word `None` right below it. This adds ugly, unnecessary clutter to your output. So, we keep it as `df.info()`.

### Why call `sum_dup(df)`, `sum_whitespace(df)`, and `sum_nulls(df)`?
- These functions are designed to handle the heavy lifting (computing counts, percentages, and displaying DataFrames). 
- **Note:** These are currently using `display()` internally to render the output as pretty tables. It's perfect to just call them without a `print` wrapper, because the `display()` call inside them handles the output.

In [45]:
def data_quality_report(df):
    """Prints a comprehensive summary of data quality for a DataFrame (Info, Duplicates, Whitespace, Nulls)."""
    
    
    print("--- Data Quality Report ---")
    print("\n" + "="*50 + "\n")
    print('---shape----')
    print(df.shape)
    print("\n" + "="*50 + "\n")
    print('---info---')
    df.info()
    print("\n" + "="*50 + "\n")
    print("--- Duplicate Check ---")
    sum_dup(df)
    print("\n" + "="*50 + "\n")
    print("\n--- Whitespace Check ---")
    sum_whitespace(df)
    print("\n--- Missing Values Check ---")
    sum_nulls(df)
    